# Mess with Mesa-Geo
To overview the critical parts of Mesa-Geo this [tutorial](https://mesa-geo.readthedocs.io/stable/tutorials/intro_tutorial.html) uses the pandemic modelling approach known as a S(usceptible), I(infected) and R(ecovered) or SIR model.

Components of the model are:

**Agents**: Each agent in the model represents an individual in the population. Agents have states of susceptible, infected, recovered, or dead. The Agents are point agents, randomly placed into the environment.

**Environment**: The environment is a set of polygons of a few Toronto neighborhoods.

**Interaction Rules**: Susceptible agents can become infected with a certain probability, if they come into contact with infected agents. Infected agents then recover after a certain period or perish based on a probability.

**Parameters**:

- Population Size (number of human agents in the model)

- Initial Infection (percent of the population initial infected)

- Exposure Distance (proximity suscpetible agents must be to infected agents to possibly get infected)

- Infection Risk (probability of becoming infected)

- Recovery Rate (time infection lasts)

- Mobility (distance agent moves)

## Set Up

### Check & Upgrade Mesa Version

In [20]:
# Show the latest version of Mesa on conda-forge
!conda search -c conda-forge mesa | tail -1
# !conda search -c conda-forge mesa 

mesa                           3.2.0    pyhd8ed1ab_0  conda-forge         


In [ ]:
%%capture
!conda install -c conda-forge -y mesa
# !conda install -c conda-forge mesa=3.2.0

In [1]:
# Version installed
import mesa
print(mesa.__version__)

3.2.0


### Verify Availble Versions from Different Sources

In [19]:
# versions available in conda-forge
!conda search -c conda-forge mesa-geo

Loading channels: done
# Name                       Version           Build  Channel             
mesa-geo                       0.5.0    pyhd8ed1ab_0  conda-forge         
mesa-geo                       0.6.0    pyhd8ed1ab_0  conda-forge         
mesa-geo                       0.7.0    pyhd8ed1ab_0  conda-forge         
mesa-geo                       0.7.1    pyhd8ed1ab_0  conda-forge         


In [19]:
# versions available in PyPI
!pip index versions mesa-geo

mesa-geo (0.9.1)
Available versions: 0.9.1, 0.9.0, 0.8.1, 0.8.0, 0.7.1, 0.7.0, 0.6.0, 0.5.0, 0.4.0, 0.3.0, 0.2.0, 0.1.2, 0.1.1, 0.1.0
  INSTALLED: 0.7.1
  LATEST:    0.9.1


### Install Mesa-Geo
Use `pip` to install the latest version(0.9.1) from PyPI.

In [ ]:
%%capture
!pip install -U mesa-geo

In [24]:
import mesa_geo
print(mesa_geo.__version__)

0.9.1


In [18]:
# Tests
import mesa.visualization, mesa.experimental

# print(dir(mesa.visualization))
print(dir(mesa.experimental))

['JupyterViz', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'jupyter_viz', 'make_text']


`ipyleaflet` is needed for the interface.

In [42]:
# %%capture
!conda install -c conda-forge -y ipyleaflet

Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done

# All requested packages already installed.



## Import Dependencies

In [2]:
# GIS library for object in the cartesian plane. 
# From Shapely we specifically need the Point class to create our human agents
from shapely.geometry import Point
# Parent ABM library to Mesa-Geo
import mesa
# import the visualization part of the library
from mesa.visualization import SolaraViz, make_plot_component
import mesa_geo as mg
from mesa_geo.visualization import make_geospace_component

## Agents

### Person
`PersonAgent`represents one human being and we initialize each person agent, it inherit from the mesa-geo librarary through the “mg.GeoAgent” in the class instantiation.  There are some required attributes for any `GeoAgent` in Mesa-Geo:
- **model**: Model object class that we will build later, this is a pointer to the model instance so the agent can get information from the model as it behaves
- **geometry**: GIS geometric object in this case a GIS Point
- **crs**: A string describing the coordinate reference system the agent is using

Attributes unique to our SIR model:
- **atype**: A string which describes the agent state (susceptible, infected, recovered, or dead)
- **mobility_range**: Distance the agent can move in meters
- **infection_risk**: A float from 0.0 to 1.0 that determines the risk of the agent being infected if exposed
- **recovery_rate**: A float from 0.0 to 1.0 that determine how long the agents takes to recover
- **death_risk**: A float from 0.0 to 1.0 that determines the probability the agent will die

The `__repr__` function is a Python primitive that will print out information as directed by the code. In this case we will print out the agent ID.

In [3]:
class PersonAgent(mg.GeoAgent):
    """Person Agent."""

    def __init__(
        self,
        model,
        geometry,
        crs,
        agent_type,
        mobility_range,
        infection_risk,
        recovery_rate,
        death_risk,
    ):
        super().__init__(model, geometry, crs)
        # Agent attributes
        self.atype = agent_type
        self.mobility_range = mobility_range
        self.infection_risk = infection_risk
        self.recovery_rate = recovery_rate
        self.death_risk = death_risk
        self.steps_infected = 0
        self.steps_recovered = 0

    def __repr__(self):
        return f"Person {self.unique_id}"
        
    # Helper function for moving agent
    def move_point(self, dx, dy):
        """
        Move a point by creating a new one
        :param dx:  Distance to move in x-axis
        :param dy:  Distance to move in y-axis
        """
        return Point(self.geometry.x + dx, self.geometry.y + dy)
        
    def step(self):
        # print(repr(self))
        # print(self.atype, self.death_risk, self.recovery_rate)

        # Part 1 - find neighbors based on infection distance
        if self.atype == "susceptible":
            neighbors = self.model.space.get_neighbors_within_distance(
                self, self.model.exposure_distance
            )
            for neighbor in neighbors:
                if (
                    neighbor.atype == "infected"
                    and self.random.random() < self.model.infection_risk
                ):
                    self.atype = "infected"
                    break  # stop process if agent becomes infected

        # Part -2 If infected, check if agent recovers or agent dies
        elif self.atype == "infected":
            if self.steps_infected >= self.recovery_rate:
                self.atype = "recovered"
                self.steps_infected = 0
            elif self.random.random() < self.death_risk:
                self.atype = "dead"
            else:
                self.steps_infected += 1

        elif self.atype == "recovered":
            self.steps_recovered += 1
            if self.steps_recovered >= 2:
                self.atype = "susceptible"
                self.steps_recovered = 0

        # Part 3 - If not dead, move
        if self.atype != "dead":
            move_x = self.random.randint(-self.mobility_range, self.mobility_range)
            move_y = self.random.randint(-self.mobility_range, self.mobility_range)
            self.geometry = self.move_point(move_x, move_y)  # Reassign geometry

        self.model.counts[self.atype] += 1  # Count agent type

#### SIR Dynamics 
The step function logic uses the agent’s `atype` to determine what actions to process.
- `susceptible`: identify all PersonAgent’s neighbors within the exposure distance with `get_neighbors_within_distance`. The step function then iterates through the list of neighbors to see if any agents are infected. If so it does a probabilistic comparison of a random float compared to the agents infection risk and if `True` the agent becomes infected and the iteration ends.

- `infected`: First, it sees how many steps the agent has been infected with `steps_infected`. If the steps are greater than or equal to their recovery rate, the agent is recovered, if not then the function does a probabilistic comparison with the agents death risk to see if the agent dies. If neither of these things happen the `steps_infected` increases by one.

-  Not `dead`: agent moves. For this we randomly get an integer for the x any (lat and long) between their negative `mobility_range` and positive `mobility` range. We pass these two integers into the helper function `move_point` and then update the agents geometry with this new point.

### Neighbourhood
`NeighbourhoodAgent` represents one geographical area as defined by the <u>geojson</u> file.

![Toronto Neighbourhood](./data/TorontoNeighbourhoods.png)

- **agent_type**: A string which describes the state of the neighborhood which will be either safe or hot spot
- **hotspot_threshold**: An integer that is the number of infected people in a neighborhood to call it a hotspot

In [4]:
class NeighbourhoodAgent(mg.GeoAgent):
    """Neighbourhood agent. Changes color according to number of infected inside it."""

    def __init__(self, model, geometry, crs, agent_type="safe", hotspot_threshold=1):
        super().__init__(model, geometry, crs)
        self.atype = agent_type
        self.hotspot_threshold = (
            hotspot_threshold  # When a neighborhood is considered a hot-spot
        )

    def __repr__(self):
        return f"Neighbourhood {self.HOODNUM}"

    def color_hotspot(self):
        # Decide if this region agent is a hot-spot
        # (if more than threshold person agents are infected)
        neighbors = self.model.space.get_intersecting_agents(self)
        infected_neighbors = [
            neighbor for neighbor in neighbors if neighbor.atype == "infected"
        ]
        if len(infected_neighbors) > self.hotspot_threshold:
            self.atype = "hotspot"
        else:
            self.atype = "safe"

    def step(self):
        """Advance agent one step."""
        # print(repr(self))
        self.color_hotspot()
        self.model.counts[self.atype] += 1  # Count agent type

#### Infection Heatmap
`NeighbourhoodAgent` change color based on the number of infected PersonAgents in their neighbourhood.

To do this we will create a helper function called `color_hotspot`. We will then use mesa-geo’s `get_intersecting_agents` function. We will then iterate through that list to get the agents with `atype` infected if the list is longer than our `hotspot_threshold` equal to 1 (so if two agents in the neighborhood are infected) then the `atype` will change to `hotspot`.

We then update our model counts.

## Model
The Model has following components: 
- A GIS layer of polygons that creates NeighbourhoodAgents from our GeoJSON file
- A diverse population of GIS Point objects, with different infection, recovery and death risks

It initializes these agents, a GIS space and step function to execute the simulation with following inital parameters:
- **pop_size**: An integer that determines the number of person agents
- **init_infection**: A float between 0.0 and 1.0 which determines what percentage of the population is infected as the model initiates
- **exposure_dist**: An integer for the distance in meters a susceptible person agent must within to be infected by a person agent who is infected
- **max_infection_risk**: A float between 0.0 and 1.0 of which determines the highest suscpetibility rate in the population

In [5]:
class GeoSIR(mesa.Model):
    """Model class for a simplistic infection model."""

    # Geographical parameters for desired map
    geojson_regions = "data/TorontoNeighbourhoods.geojson"

    def __init__(
        self,
        pop_size=30,
        mobility_range=500,
        init_infection=0.2,
        exposure_dist=500,
        max_infection_risk=0.2,
        max_recovery_time=5,
    ):
        super().__init__()
        self.space = mg.GeoSpace(warn_crs_conversion=False)
        # track agent types(e.g. infected)
        self.counts = None  
        self.reset_counts()  

        # SIR model parameters
        self.pop_size = pop_size
        self.mobility_range = mobility_range
        self.initial_infection = init_infection
        self.exposure_distance = exposure_dist
        self.infection_risk = max_infection_risk
        self.recovery_rate = max_recovery_time
        self.running = True  
        self.datacollector = mesa.DataCollector(
            {
                "infected": get_infected_count,
                "susceptible": get_susceptible_count,
                "recovered": get_recovered_count,
                "dead": get_dead_count,
                "safe": get_safe_count,
                "hotspot": get_hotspot_count,
            }
        )

        # Set up the Neighbourhood patches for every region in file
        ac = mg.AgentCreator(NeighbourhoodAgent, model=self)
        neighbourhood_agents = ac.from_file(self.geojson_regions)

        # Add neighbourhood agents to space
        self.space.add_agents(neighbourhood_agents)

        # Generate random location, add agent to grid
        for i in range(pop_size):
            # assess if they are infected
            if self.random.random() < self.initial_infection:
                agent_type = "infected"
            else:
                agent_type = "susceptible"
            # determine movement range
            mobility_range = self.random.randint(0, self.mobility_range)
            # determine agent recovery rate
            recover = self.random.randint(1, self.recovery_rate)
            # determine agents infection risk
            infection_risk = self.random.uniform(0, self.infection_risk)
            # determine agent death probability
            death_risk = self.random.random()

            # Generate PersonAgent population
            unique_person = mg.AgentCreator(
                PersonAgent,
                model=self,
                crs=self.space.crs,
                agent_kwargs={
                    "agent_type": agent_type,
                    "mobility_range": mobility_range,
                    "recovery_rate": recover,
                    "infection_risk": infection_risk,
                    "death_risk": death_risk,
                },
            )

            x_home, y_home = self.find_home(neighbourhood_agents)

            this_person = unique_person.create_agent(Point(x_home, y_home))
            self.space.add_agents(this_person)
            
    def find_home(self, neighbourhood_agents):
        """Find start location of agent"""

        # identify location
        this_neighbourhood = self.random.randint(
            0, len(neighbourhood_agents) - 1
        )  # Region where agent starts
        center_x, center_y = neighbourhood_agents[
            this_neighbourhood
        ].geometry.centroid.coords.xy
        this_bounds = neighbourhood_agents[this_neighbourhood].geometry.bounds
        spread_x = int(
            this_bounds[2] - this_bounds[0]
        )  # Heuristic for agent spread in region
        spread_y = int(this_bounds[3] - this_bounds[1])
        this_x = center_x[0] + self.random.randint(0, spread_x) - spread_x / 2
        this_y = center_y[0] + self.random.randint(0, spread_y) - spread_y / 2

        return this_x, this_y

    def reset_counts(self):
        self.counts = {
            "susceptible": 0,
            "infected": 0,
            "recovered": 0,
            "dead": 0,
            "safe": 0,
            "hotspot": 0,
        }

    def step(self):
        """Run one step of the model."""
        self.reset_counts()
        # Activate PersonAgents in random order
        self.agents_by_type[PersonAgent].shuffle_do("step")
        # For NeighbourhoodAgents the order doesn't matter, since they update independently from each other
        self.agents_by_type[NeighbourhoodAgent].do("step")
        self.datacollector.collect(self)  
        # Run until no one is infected
        if self.counts["infected"] == 0:
            self.running = False

### Helper Functions
`find_home`: As Mesa-Geo is an GIS based ABM, we need assign each PersonAgent a Geometry and location. This helper function first identifies a NeighbourhoodAgent where the PersonAgent will start. Next it identifies the center of the neighborhood and its boundary and then randomly moving from the center point, put staying within the bounds, it a lat and long to aissgn the PersonAgent is starting location.

In [6]:
# Functions needed for datacollector
def get_infected_count(model):
    return model.counts["infected"]


def get_susceptible_count(model):
    return model.counts["susceptible"]


def get_recovered_count(model):
    return model.counts["recovered"]


def get_dead_count(model):
    return model.counts["dead"]


def get_hotspot_count(model):
    return model.counts["hotspot"]


def get_safe_count(model):
    return model.counts["safe"]

## Run the Model

In [7]:
model = GeoSIR()
for i in range(5):
    model.step()

model.datacollector.get_model_vars_dataframe()

/opt/miniconda3/envs/my_env/lib/python3.11/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


,infected,susceptible,recovered,dead,safe,hotspot
0,2,22,0,6,5,0
1,2,22,0,6,5,0
2,0,22,2,6,5,0
3,0,22,2,6,5,0
4,0,24,0,6,5,0


## Interface

### Define Agent Portayal

In [11]:
def SIR_draw(agent):
    """
    Portrayal Method for canvas
    """
    portrayal = {}
    if isinstance(agent, PersonAgent):
        if agent.atype == "susceptible":
            portrayal["color"] = "Green"
        elif agent.atype == "infected":
            portrayal["color"] = "Red"
        elif agent.atype == "recovered":
            portrayal["color"] = "Blue"
        else:
            portrayal["marker_type"] = "AwesomeIcon" # death
            portrayal["name"] = "times"
            portrayal["icon_properties"] = {
                "marker_color": "black",
                "icon_color": "white",
            }

    if isinstance(agent, NeighbourhoodAgent):
        if agent.atype == "hotspot":
            portrayal["color"] = "Red"
        else:
            portrayal["color"] = "Green"

    return portrayal

### Set Sliders for Model Parameters

In [9]:
model_params = {
    "pop_size": {
        "type": "SliderInt",
        "value": 80,
        "label": "Population Size",
        "min": 0,
        "max": 100,
        "step": 1,
    },
    "mobility_range": {
        "type": "SliderInt",
        "value": 500,
        "label": "Max Possible Agent Movement",
        "min": 100,
        "max": 1000,
        "step": 50,
    },
    "init_infection": {
        "type": "SliderFloat",
        "value": 0.4,
        "label": "Initial Infection",
        "min": 0.0,
        "max": 1.0,
        "step": 0.1,
    },
    "exposure_dist": {
        "type": "SliderInt",
        "value": 800,
        "label": "Exposure Distance",
        "min": 100,
        "max": 1000,
        "step": 50,
    },
    "max_infection_risk": {
        "type": "SliderFloat",
        "value": 0.7,
        "label": "Maximum Infection Risk",
        "min": 0.0,
        "max": 1.0,
        "step": 0.1,
    },
    "max_recovery_time": {
        "type": "SliderInt",
        "value": 7,
        "label": "Maximum Number of Steps to Recover",
        "min": 1,
        "max": 10,
        "step": 1,
    },
}

### Call the Model through Mesa-Geo visualization Module

In [10]:
model = GeoSIR()
page = SolaraViz(
    model,
    name="GeoSIR",
    model_params=model_params,
    components=[
        make_geospace_component(SIR_draw, zoom=12, scroll_wheel_zoom=False),
        make_plot_component(["infected", "susceptible", "recovered", "dead"]),
        make_plot_component(["safe", "hotspot"]),
    ],
)
# This is required to render the visualization in the Jupyter notebook
page

Cannot show ipywidgets in text

/opt/miniconda3/envs/my_env/lib/python3.11/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)
/opt/miniconda3/envs/my_env/lib/python3.11/site-packages/shapely/predicates.py:778: RuntimeWarning: invalid value encountered in intersects
  return lib.intersects(a, b, **kwargs)


### Tests

In [1]:
from ipyleaflet import Map
Map(center=(35.4437, 139.6380), zoom=10)  # Example: Yokohama

Map(center=[35.4437, 139.638], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zo…